In [52]:
require "dotenv"
Dotenv.overload(File.join(__dir__, ".env"))

{}

In [53]:
# gem install aws-sdk-s3vectors
# gem install aws-sdk --verbose
# gem install nokogiri loofah

In [56]:
require "aws-sdk"
require "yaml"
require "active_support"
require "csv"
require "reverse_markdown"
#each row has a content field andd each content field has a content as well, its content we need to get all the body and its html
require "nokogiri"
require "loofah"
require "active_support/core_ext/hash/indifferent_access"
require "active_support/time"   # if you also had TimeWithZone

false

In [57]:
s3 = Aws::S3::Client.new(region: ENV.fetch("AWS_REGION"))

#<Aws::S3::Client>

In [58]:
# test that we have authorization
# resp = s3.list_buckets
# puts "Buckets in account:"
# resp.buckets.each { |b| puts "- #{b.name}" }

In [59]:
#load the json file
file_path = File.join(__dir__, "bedrock", "corporate.csv")

"./bedrock/corporate.csv"

In [60]:
# Read CSV into an array of rows (each row is a CSV::Row object)

rows = CSV.read(file_path, headers: true).map{|r| r.to_h};nil

In [61]:
# yaml parsing function

def parse_yaml(yaml_text)
    begin
    parsed = YAML.safe_load(
      yaml_text,
      permitted_classes: [Time, Date, ActiveSupport::HashWithIndifferentAccess, ActiveSupport::TimeWithZone, ActiveSupport::TimeZone, ActiveSupport::Duration],  # allow Time and Date
      aliases: true
    )
    rescue Psych::SyntaxError => e
      puts "YAML parsing error: #{e.message}"
    end
end

:parse_yaml

In [62]:
# Follow nested "content" keys until the terminal payload
# e.g. row["content"] -> YAML Hash with key "content" -> ... -> HTML/String
def dig_content(payload)
  current = payload
  # Keep drilling as long as it's a Hash with a "content" key
  while current.is_a?(Hash) && current.key?("resource")
    current = current["resource"]["content"]
  end
  current
end


:dig_content

In [63]:
# Return ONLY the <body> inner HTML (if present). If no <body>, return full HTML.
def extract_body_html(html_str)
  doc = Nokogiri::HTML(html_str.to_s)
  body = doc.at("body")
  (body ? body.inner_html : doc.inner_html).to_s
end

# Sanitize body HTML: strip scripts/styles, JS events, and unsafe URL schemes
def sanitize_body_html(body_html)
  # Scrub to a conservative but useful subset (you can relax this if needed)
  Loofah.fragment(body_html)
        .scrub!(:prune) # removes script/style/iframe/object, etc.
        .scrub!(Loofah::Scrubber.new do |node|
          # remove on* event attributes and javascript: URLs
          if node.element?
            node.attribute_nodes.each do |attr|
              if attr.name.downcase.start_with?("on")
                attr.remove
              elsif %w[href src].include?(attr.name.downcase)
                attr.remove if attr.value.to_s.strip.downcase.start_with?("javascript:")
              end
            end
          end
        end)
        .to_html
end

# Convenience: from YAML text → sanitized body HTML
def content_to_sanitized_body_html(yaml_text)
  parsed    = parse_yaml(yaml_text)
    parsed.map do |components|
          content   = dig_content(comp) # should now be HTML (String) or nil
          raw_body  = extract_body_html(content)
          sanitize_body_html(raw_body)
    end

rescue  => e
    puts e.message
end

:content_to_sanitized_body_html

In [64]:
# --- HTML → safe HTML ---
def sanitize_html(html_str)
  frag = Loofah.fragment(html_str.to_s)

  # Remove obvious junk: script/style/iframe/object etc.
  frag.scrub!(:prune)

  # Strip event handlers + javascript: URLs
  frag.scrub!(Loofah::Scrubber.new do |node|
    next unless node.element?
    node.attribute_nodes.each do |attr|
      name = attr.name.downcase
      if name.start_with?("on") || %w[href src].include?(name) && attr.value.to_s.strip.downcase.start_with?("javascript:")
        attr.remove
      end
    end
  end)

  # Remove empty nodes that are just layout noise (<div></div>, <p>&nbsp;</p>, etc.)
  frag.scrub!(Loofah::Scrubber.new do |node|
    next unless node.element?
    html = node.inner_text.gsub(/\u00A0/, " ").strip
    node.remove if html.empty? && node.children.empty?
  end)

  # Normalize nbsp and collapse whitespace
  html = frag.to_html
  html = html.gsub(/\u00A0/, " ")
  html = html.gsub(/[ \t]+/, " ")

  html
end

# --- Safe HTML → Markdown (structure preserved for LLMs) ---
def html_to_markdown(html)
  ReverseMarkdown.convert(
    html,
    unknown_tags: :drop,    # drop weird tags
    github_flavored: true,  # nicer lists/code handling
    tag_border: ""          # avoid extra spaces
  )
end

# --- One-shot: HTML/Array → cleaned Markdown ---
def sanitize_for_llm(body)
  # Normalize: content may be a String or an Array of fragments
  fragments = body.is_a?(Array) ? body : [body]

  cleaned_md = fragments.map do |frag|
    safe_html = sanitize_html(frag)
    next nil if safe_html.strip.empty? || safe_html.strip == "<div></div>" # trivial bodies
    md = html_to_markdown(safe_html)

    # Final text normalization: collapse blank lines, trim
    md = md.gsub(/[ \t]+\n/, "\n")
           .gsub(/\n{3,}/, "\n\n")
           .strip
    md.empty? ? nil : md
  end.compact

  # Join fragments with a separator the LLM can use
  cleaned_md.join("\n\n---\n\n")
end


:sanitize_for_llm

In [67]:
ITG_URL = "https://corporate.itglue.com"
test_rows = rows;nil
bodies = []
test_rows.each_with_index do |row, i|
  yaml_text = row["content"]
  next unless yaml_text

  begin
    parsed = parse_yaml(yaml_text)
    mds = []
    parsed.each do |comp|
        body   = dig_content(comp)  # can be String or Array like your sample
        md     = sanitize_for_llm(body)
        mds << md
        if md.nil? || md.empty?
          puts "Row #{i}: (empty after sanitation)" if (i % 500).zero?
        else
          puts "Row #{i} LLM-ready markdown:\n#{}\n---" if (i % 500).zero?
        end
    end;nil
      bodies << {
            markdown: mds,
            resource_id: row["id"],
            name: row["name"],
            url: "#{ITG_URL}/documents/#{row["id"]}"
        }
  rescue Psych::SyntaxError => e
    warn "Row #{i} YAML error: #{e.message}"
  end
end;nil

(irb): warning: already initialized constant Object::ITG_URL


Row 0: (empty after sanitation)
Row 1000 LLM-ready markdown:

---
Row 1000 LLM-ready markdown:

---
Row 1000 LLM-ready markdown:

---
Row 1000 LLM-ready markdown:

---
Row 1000 LLM-ready markdown:

---
Row 1000 LLM-ready markdown:

---
Row 1000 LLM-ready markdown:

---
Row 1000 LLM-ready markdown:

---
Row 1000 LLM-ready markdown:

---
Row 1500 LLM-ready markdown:

---
Row 2000 LLM-ready markdown:

---
Row 2000 LLM-ready markdown:

---
Row 2000 LLM-ready markdown:

---
Row 2000 LLM-ready markdown:

---
Row 2500 LLM-ready markdown:

---
Row 3000 LLM-ready markdown:

---
Row 3000 LLM-ready markdown:

---
Row 3000 LLM-ready markdown:

---
Row 3000 LLM-ready markdown:

---
Row 3000 LLM-ready markdown:

---
Row 3000 LLM-ready markdown:

---
Row 3000 LLM-ready markdown:

---
Row 3000 LLM-ready markdown:

---
Row 3000 LLM-ready markdown:

---


In [ ]:
# test_rows = rows[0..10];nil
# bodies = []
# test_rows.each_with_index do |row, i|
#   yaml_text = row["content"]
#   next unless yaml_text

#   begin
#     safe_body_html = content_to_sanitized_body_html(yaml_text)
#       bodies << safe_body_html
#     puts "Row #{i} sanitized body HTML:\n#{safe_body_html}\n---"
#   rescue Psych::SyntaxError => e
#     warn "Row #{i} YAML error: #{e.message}"
#   end
# end

In [47]:
bodies[0..20]

[{:markdown=>[""], :resource_id=>"2916", :name=>"IT Glue Demo - Alpha Technologies (2014-12)"}, {:markdown=>[""], :resource_id=>"2918", :name=>"IT Glue Demo - Aldridge (2014-12)"}, {:markdown=>[""], :resource_id=>"3112", :name=>"IT Glue Demo - LSeven (2014-12)"}, {:markdown=>["T shirts and promo materials", "One big leading question on the back of the T:\n\n- is your documentation a dog's breakfast?\n- how much do you trust your own words?\n-"], :resource_id=>"706", :name=>"IT Nation prep"}, {:markdown=>[""], :resource_id=>"2940", :name=>"BizDox Webinar (2014-12)"}, {:markdown=>["", "", "BizDox", "Bizdox forum notes to demonstrate approach/issues", "Bizdocs ideas/suggestions to consider in IT Glue", "What makes IT Glue stand out from Bizdox?", "BizDox Screenshots", "BizDox Pros", "BizDox Cons", "- Business-focus vs. tech focus\n- Techs not using it... or putting data in... why not?\n- Visually heavy... not fast and furious\n- Culture of continuous improvement\n- Collaboration is just n

In [68]:
bodies.count

3150

In [69]:
# Save to JSON
OUTPUT =  File.join(__dir__, "bedrock", "sanitized_rows.json")
File.open(OUTPUT, "w") do |f|
  f.write(JSON.pretty_generate(bodies))
end

puts "Saved #{bodies.size} rows to #{OUTPUT}"

(irb):1: warning: already initialized constant Object::OUTPUT
(irb):1: warning: previous definition of OUTPUT was here


Saved 3150 rows to ./bedrock/sanitized_rows.json
